In [7]:
import copy
import math
import torch

class EarlyStopping:
    """
    Stop training when the monitored metric hasn't improved after `patience` epochs.
    - monitor: 'loss' (minimize) or 'acc' (maximize)
    - min_delta: minimum change to qualify as an improvement (absolute, not relative)
    - restore_best_weights: if True, loads the best model weights on stop
    - checkpoint_path: if set, saves best model state_dict here on improvement
    """
    def __init__(self, patience=5, monitor='loss', min_delta=0.0,
                 restore_best_weights=True, checkpoint_path=None):
        assert monitor in ('loss', 'acc')
        self.patience = patience
        self.monitor = monitor
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.checkpoint_path = checkpoint_path

        self.best_score = None
        self.best_state = None
        self.count = 0
        self.should_stop = False

    def _is_improvement(self, current, best):
        if self.monitor == 'loss':
            # improvement = lower is better
            return (best - current) > self.min_delta
        else:
            # improvement = higher is better
            return (current - best) > self.min_delta

    def step(self, model, val_loss, val_acc):
        current = val_acc if self.monitor == 'acc' else val_loss
        # Initialize best
        if self.best_score is None:
            self.best_score = current
            if self.restore_best_weights:
                self.best_state = copy.deepcopy(model.state_dict())
            if self.checkpoint_path:
                torch.save(model.state_dict(), self.checkpoint_path)
            self.count = 0
            return

        if self._is_improvement(current, self.best_score):
            self.best_score = current
            if self.restore_best_weights:
                self.best_state = copy.deepcopy(model.state_dict())
            if self.checkpoint_path:
                torch.save(model.state_dict(), self.checkpoint_path)
            self.count = 0
        else:
            self.count += 1
            if self.count >= self.patience:
                self.should_stop = True

    def finalize(self, model):
        if self.restore_best_weights and self.best_state is not None:
            model.load_state_dict(self.best_state)


In [11]:
# -*- coding: utf-8 -*-
import math, random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --------------------
# 1) Load & prepare data
# --------------------
PAD, SOS, EOS = "<pad>", "<s>", "</s>"

class PairDataset(Dataset):
    def __init__(self, path):
        self.pairs = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                # tab-separated: col0=input, col1=target
                cols = line.split("\t")
                if len(cols) < 2:
                    continue
                src, tgt = cols[0].strip(), cols[1].strip()
                self.pairs.append((src, tgt))

        # build vocab from all tokens (space-separated)
        def tok(s): return s.split()
        all_tokens = [t for s,t in self.pairs for t in (tok(s)+tok(t))]
        # unique while preserving order
        seen = set()
        vocab_list = [PAD, SOS, EOS]
        for w in all_tokens:
            if w not in seen:
                seen.add(w); vocab_list.append(w)
        self.itos = vocab_list
        self.stoi = {w:i for i,w in enumerate(self.itos)}

    def __len__(self): return len(self.pairs)

    def encode(self, s, add_sos=False, add_eos=True):
        ids = [self.stoi.get(w, None) for w in s.split()]
        ids = [i for i in ids if i is not None]
        if add_sos: ids = [self.stoi[SOS]] + ids
        if add_eos: ids = ids + [self.stoi[EOS]]
        return ids

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return src, tgt

def collate(batch, ds: PairDataset, max_len=128):
    src_seqs, tgt_seqs = [], []
    for src, tgt in batch:
        src_ids = ds.encode(src, add_sos=False, add_eos=True)[:max_len]
        tgt_ids = ds.encode(tgt, add_sos=True,  add_eos=True)[:max_len]  # decoder expects SOS
        src_seqs.append(src_ids)
        tgt_seqs.append(tgt_ids)

    # pad
    pad_id = ds.stoi[PAD]
    def pad_to_max(seqs):
        L = max(len(s) for s in seqs)
        return torch.tensor([s + [pad_id]*(L-len(s)) for s in seqs], dtype=torch.long)
    return pad_to_max(src_seqs), pad_to_max(tgt_seqs)

# --------------------
# 2) Model: Encoder-Decoder (GRU)
# --------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.rnn = nn.GRU(d_model, d_model, num_layers=n_layers, batch_first=True)
    def forward(self, x, lengths=None):
        x = self.emb(x)
        outputs, h = self.rnn(x)   # outputs: (B,T,H), h: (L,B,H)
        return outputs, h

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.rnn = nn.GRU(d_model, d_model, num_layers=n_layers, batch_first=True)
        self.fc  = nn.Linear(d_model, vocab_size)
    def forward(self, x, h):
        # x: (B,1) next token ids
        emb = self.emb(x)
        out, h = self.rnn(emb, h)
        logits = self.fc(out)  # (B,1,V)
        return logits, h

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, pad_id):
        super().__init__()
        self.enc = encoder
        self.dec = decoder
        self.pad_id = pad_id

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """
        src: (B, S)    (EoS-terminated)
        tgt: (B, T)    (Sos ... EoS)
        returns: logits (B, T-1, V)
        """
        B, T = tgt.size()
        _, h = self.enc(src)                 # h: (L,B,H); use final hidden for decoder init
        V = self.dec.fc.out_features

        # we will predict tokens 1..T-1 given inputs 0..T-2
        logits_out = []
        y = tgt[:, 0:1]  # first token is SOS
        for t in range(1, T):
            logits, h = self.dec(y, h)      # logits: (B,1,V)
            logits_out.append(logits)
            use_teacher = random.random() < teacher_forcing_ratio
            y = tgt[:, t:t+1] if use_teacher else logits.argmax(-1)
        return torch.cat(logits_out, dim=1)

# --------------------
# 3) Train
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = Path("DataTrain1.txt")
ds = PairDataset(data_path)

# split
random.seed(0)
idx = list(range(len(ds)))
random.shuffle(idx)
split = int(0.9*len(idx))
train_ids, val_ids = idx[:split], idx[split:]

class SubsetWrap(Dataset):
    def __init__(self, base, ids): self.base, self.ids = base, ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.base[self.ids[i]]

train_set, val_set = SubsetWrap(ds, train_ids), SubsetWrap(ds, val_ids)
collate_fn = lambda b: collate(b, ds)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_set, batch_size=64, shuffle=False, collate_fn=collate_fn)

vocab_size = len(ds.itos)
pad_id = ds.stoi[PAD]
encoder = Encoder(vocab_size, d_model=256)
decoder = Decoder(vocab_size, d_model=256)
model = Seq2Seq(encoder, decoder, pad_id).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

from tqdm import tqdm

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, total_tok = 0.0, 0
    total_correct = 0
    loop = tqdm(loader, desc="Train" if train else "Val", leave=False)

    for src, tgt in loop:
        src, tgt = src.to(device), tgt.to(device)

        with torch.set_grad_enabled(train):
            logits = model(src, tgt, teacher_forcing_ratio=0.5 if train else 0.0)
            target = tgt[:, 1:]  # shift left
            loss = criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        # mask out padding
        mask = (target != pad_id)
        preds = logits.argmax(-1)
        correct = ((preds == target) & mask).sum().item()
        total_correct += correct
        total_tok += mask.sum().item()
        total_loss += loss.item() * mask.sum().item()

        loop.set_postfix(loss=loss.item(),
                         acc=100.0 * total_correct / max(total_tok, 1))

    avg_loss = total_loss / max(total_tok, 1)
    avg_acc = total_correct / max(total_tok, 1)
    return avg_loss, avg_acc


# Training with progress bars
# choose what to monitor: 'loss' (minimize) or 'acc' (maximize)
early_stopper = EarlyStopping(
    patience=10,
    monitor='loss',          # or 'acc'
    min_delta=1e-3,          # require at least this much improvement
    restore_best_weights=True,
    checkpoint_path=None       #"best_seq2seq.pt"  optional; set None to skip saving
)

EPOCHS = 50  # set a high cap; early stopping will cut it short
for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)

    # Perplexity is optional but nice:
    tr_ppl = math.exp(train_loss) if train_loss < 20 else float('inf')
    va_ppl = math.exp(val_loss)   if val_loss   < 20 else float('inf')

    print(f"Epoch {epoch:02d} | "
          f"train CE/token {train_loss:.4f} (PPL {tr_ppl:.2f}), acc {train_acc:.2%} | "
          f"val CE/token {val_loss:.4f} (PPL {va_ppl:.2f}), acc {val_acc:.2%}")

    # Early stopping step on validation metrics
    early_stopper.step(model, val_loss=val_loss, val_acc=val_acc)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

# Restore best weights (if enabled)
early_stopper.finalize(model)

# If you set checkpoint_path, you can also reload explicitly later:
# model.load_state_dict(torch.load("best_seq2seq.pt", map_location=device))

# --------------------
# 4) Quick inference helper
# --------------------
itos, stoi = ds.itos, ds.stoi

def greedy_decode(sentence, max_len=64):
    model.eval()
    with torch.no_grad():
        src = torch.tensor([ds.encode(sentence, add_eos=True)], dtype=torch.long, device=device)
        _, h = model.enc(src)

        y = torch.tensor([[stoi[SOS]]], dtype=torch.long, device=device)
        out_ids = []
        for _ in range(max_len):
            logits, h = model.dec(y, h)
            next_id = logits.argmax(-1)  # (1,1)
            token_id = int(next_id.item())
            if token_id == stoi[EOS] or token_id == stoi[PAD]:
                break
            out_ids.append(token_id)
            y = next_id
        return " ".join(itos[i] for i in out_ids)

# Test model
def test_model(sentence, max_len=50):
    model.eval()
    with torch.no_grad():
        # Encode source
        src = torch.tensor([ds.encode(sentence, add_eos=True)], dtype=torch.long, device=device)
        _, h = model.enc(src)

        # Start decoder with <s>
        y = torch.tensor([[ds.stoi["<s>"]]], dtype=torch.long, device=device)
        out_ids = []
        for _ in range(max_len):
            logits, h = model.dec(y, h)      # (1,1,V)
            next_id = logits.argmax(-1)      # pick best token
            token_id = int(next_id.item())
            if token_id in (ds.stoi["</s>"], ds.stoi["<pad>"]):
                break
            out_ids.append(token_id)
            y = next_id

        # Convert IDs back to words
        return " ".join(ds.itos[i] for i in out_ids)


Epoch 01 | train CE/token 6.2389 (PPL 512.28), acc 14.67% | val CE/token 5.8529 (PPL 348.25), acc 17.85%


Epoch 02 | train CE/token 5.5471 (PPL 256.50), acc 17.17% | val CE/token 5.8112 (PPL 334.03), acc 17.85%


Epoch 03 | train CE/token 5.2222 (PPL 185.34), acc 18.11% | val CE/token 5.7965 (PPL 329.16), acc 16.04%


Epoch 04 | train CE/token 4.8415 (PPL 126.65), acc 20.76% | val CE/token 5.5748 (PPL 263.69), acc 17.08%


Epoch 05 | train CE/token 4.5247 (PPL 92.27), acc 21.95% | val CE/token 5.5034 (PPL 245.53), acc 17.21%


...

Epoch 26 | train CE/token 0.0533 (PPL 1.05), acc 99.86% | val CE/token 2.2961 (PPL 9.93), acc 76.84%


Epoch 27 | train CE/token 0.0500 (PPL 1.05), acc 99.81% | val CE/token 2.3232 (PPL 10.21), acc 76.84%


Epoch 28 | train CE/token 0.0445 (PPL 1.05), acc 99.88% | val CE/token 2.3562 (PPL 10.55), acc 76.84%


Epoch 29 | train CE/token 0.0404 (PPL 1.04), acc 99.84% | val CE/token 2.3355 (PPL 10.33), acc 76.84%


Epoch 30 | train CE/token 0.0352 (PPL 1.04), acc 99.90% | val CE/token 2.3492 (PPL 10.48), acc 77.10%
Early stopping triggered at epoch 30.


In [12]:
# simple interactive test
while True:
    sentence = input("Enter a Vietnamese sentence (or 'quit' to stop): ")
    if sentence.lower() in ("quit", "exit", "q"):
        break
    prediction = test_model(sentence)
    print("Model prediction:", prediction)
    print()


Enter a Vietnamese sentence (or 'quit' to stop): đây là buổi chiều
Model prediction: ở đây lạ trước quen sau

Enter a Vietnamese sentence (or 'quit' to stop): mẹ của cô ấy
Model prediction: mẹ cô ấy mẹ

Enter a Vietnamese sentence (or 'quit' to stop): quit
